# Depth T of the monitored QELM: T = 10, T = N, T = 2N

Issue #268, after #261. The QELM of `experiments/haar_readouts`: a reference qubit `R` entangled with the charge sector of `N` system qubits, `T` rounds of one brickwork layer of Haar gates (one fixed circuit per seed) and a Z measurement of each system qubit with probability `p`, then a ridge readout of the observables of the final state (`<P_k>`, `<P_i P_j>`, the set `all`) that guesses the sector.

Three rules for the depth `T(N)` are compared, for `N = 6, 8, 10, 12`:

- `T = 10` for every `N` (Dehghani et al., Nature Comm. 2023);
- `T = N`, enough for the light cone of the brickwork (2 sites per round) to cover the chain;
- `T = 2N`, the stationary regime of the MIPT.

For each: the accuracy against `p` and the purification `S_R` (the order parameter of the MIPT), the steepest slope of the accuracy, the crossings of the curves of successive `N`, the cost, the variance of `<Z_i>` site by site, and the condition number of `X^T X`.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what `tests/test_haar_depth_notebook.py` does for a small sweep). The full sweep (the defaults) takes about 3 hours on 4 cores. The sweep goes to this notebook's own folder, `outputs/haar_depth/<date_time>/`, and `RUN = False` reads back the latest sweep of this folder.

In [ ]:
N_VALUES = os.environ.get("N_VALUES", "6 8 10 12")
DEPTHS = os.environ.get("DEPTHS", "10 N 2N")
SEEDS = os.environ.get("SEEDS", "42 43 44 45 46 47 48 49 50 51")
PER_P = os.environ.get("PER_P", "2000")
# Empty: the grid of depth_scaling.sh, p = 0 to 0.4.
P_VALUES = os.environ.get("P_VALUES", "")
MAX_JOBS = os.environ.get("MAX_JOBS", "4")
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "0") == "1"

## Launch the sweep

One run per `N`, depth rule and seed (`experiments/haar_readouts/depth_scaling.sh`).

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

# This notebook's own output folder (issue #151).
EXPERIMENT = "haar_depth"

if RUN:
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    env = {
        **os.environ,
        "N_VALUES": N_VALUES,
        "DEPTHS": DEPTHS,
        "SEEDS": SEEDS,
        "PER_P": PER_P,
        "MAX_JOBS": MAX_JOBS,
        "TQDM_DISABLE": "1",
        "OUT_DIR": str(sweep_dir),
    }
    if P_VALUES:
        env["P_VALUES"] = P_VALUES
    run_with_progress(
        ["bash", "experiments/haar_readouts/depth_scaling.sh"],
        total=len(N_VALUES.split()) * len(DEPTHS.split()) * len(SEEDS.split()),
        env=env,
        show=("N", "depth_rule", "seed"),
    )
else:
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Read back the runs and summarize

`summary` holds, for each rule and `N`: `T`, the sigmoid fit of the accuracy (`p0`, the width `w`, the steepest slope `a / 4w`), the simulation time of one run, and the crossings of the accuracy and `S_R` curves of successive sizes with their spread.

In [ ]:
import importlib.util

import matplotlib.pyplot as plt

from QELM.utils.plots import save_figure

spec = importlib.util.spec_from_file_location(
    "plot_depth_scaling", "experiments/haar_readouts/plot_depth_scaling.py"
)
plot = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot)

rows = plot.load_rows([sweep_dir])
summary = plot.summarize(rows)
print(f"{'rule':>5} {'N':>3} {'T':>3} {'p0':>7} {'slope':>7} {'time (s)':>9}")
for rule, info in summary["rules"].items():
    for n, s in info["sizes"].items():
        fit = s["fit"]
        print(
            f"{rule:>5} {n:>3} {s['T']:>3} {fit['p0']:7.3f} {fit['slope']:7.2f}"
            f" {s['time_simulation_s']:9.1f}"
        )
    print(f"      accuracy crossings: {[c['p_x'] for c in info['accuracy_crossings']]}")
    print(f"      S_R crossings:      {[c['p_x'] for c in info['S_R_crossings']]}")
print("sharpest accuracy crossing:", summary["sharpest_accuracy_crossing"])
print("sharpest S_R crossing:", summary["sharpest_S_R_crossing"])

## Figure 1: transition profiles, one column per rule

Top: accuracy of the QELM ridge against `p`, one curve per `N` (points: mean over the circuits, error bars: standard error; thin line: the sigmoid fit). Bottom: purification of `R`.

In [ ]:
fig, table = plot.draw_profiles(rows, summary)
figure_path = save_figure(fig, sweep_dir, "depth_profiles", data=table)
plt.show()

## Figure 2: sharpness and finite-size crossings

Left: steepest slope of the accuracy against `N`. Middle and right: crossing points of the accuracy and of the `S_R` curves of successive sizes. The rule whose crossings agree best is in `#fa5703`.

In [ ]:
fig, table = plot.draw_crossings(rows, summary)
save_figure(fig, sweep_dir, "depth_crossings", data=table)
plt.show()

## Figure 3: cost against gain

Simulation time of one run against `T`, and the steepest slope against that time.

In [ ]:
fig, table = plot.draw_cost(rows, summary)
save_figure(fig, sweep_dir, "depth_cost", data=table)
plt.show()

## Light cone: variance of `<Z_i>` site by site, largest `N`

A site outside the light cone of the rest of the chain keeps the same distribution of `<Z_i>` whatever the record; here every site spreads alike, edges included. Then the condition number of the training `X^T X` of the QELM features.

In [ ]:
for name, drawn in (
    ("depth_edges", plot.draw_edges(rows)),
    ("depth_conditioning", plot.draw_conditioning(rows)),
):
    if drawn is not None:
        fig, table = drawn
        save_figure(fig, sweep_dir, name, data=table)
        plt.show()